# A1: InternVL3-2B + QLoRA

A1 — независимая архитектурная проверка для проекта. Вместо Qwen2.5-VL здесь используется `OpenGVLab/InternVL3-2B-hf`: современная VLM с поддержкой стандартного интерфейса Transformers. Это позволяет сравнить, как две разные архитектуры адаптируются к русскоязычным инструкциям VK.

**Конфигурация A1:** те же 2 000 примеров `LLaVA-Instruct-ru`, одна эпоха, QLoRA rank=8, batch size=1 и накопление градиента=8. Последняя ячейка сразу запускает обучение и сохраняет адаптер.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import (
    AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig,
    Trainer, TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ['PYTHONIOENCODING'] = 'utf-8'
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA = ROOT / 'data' / 'raw' / 'llava_instruct_ru_train.json'
SUBSET_PATH = ROOT / 'data' / 'processed' / 'llava_instruct_ru_first_iteration_seed42.csv'
IMAGE_CACHE = ROOT / 'data' / 'coco_cache' / 'train2017'
MODEL_DIR = ROOT / 'models' / 'a1_internvl3_2b_qlora_adapter'
CHECKPOINT_DIR = ROOT / 'checkpoints' / 'a1_internvl3_2b_qlora'
RESULTS_DIR = ROOT / 'results'
for directory in (MODEL_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'OpenGVLab/InternVL3-2B-hf'
SEED = 42
TRAINING_EXAMPLES = 2_000
MAX_LENGTH = 1024
assert torch.cuda.is_available(), 'Для A1 нужна CUDA.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Та же фиксированная выборка VK

Чтобы сравнение было честным, A1 получает те же 2 000 строк, что и F1-fast и F2. Изображения берутся из уже загруженного локального кэша COCO; повторное скачивание не нужно.

In [ ]:
subset = pd.read_csv(SUBSET_PATH)
raw_records = json.loads(RAW_DATA.read_text(encoding='utf-8'))

def parse_record(record):
    messages = record['conversations']
    question = next(item['value'] for item in messages if item['from'] == 'human')
    answer = next(item['value'] for item in messages if item['from'] == 'gpt')
    return question.replace('<image>\n', '').strip(), answer.strip(), record['image']

rows = []
for row in subset[['row_number', 'type']].itertuples(index=False):
    question, answer, image_path = parse_record(raw_records[int(row.row_number)])
    rows.append({'type': row.type, 'question': question, 'answer': answer, 'image_path': image_path})
full_train_df = pd.DataFrame(rows)
train_df = (full_train_df.groupby('type', group_keys=False)
            .apply(lambda group: group.sample(
                n=round(TRAINING_EXAMPLES * len(group) / len(full_train_df)), random_state=SEED))
            .reset_index(drop=True))
train_df = train_df.sample(n=TRAINING_EXAMPLES, random_state=SEED).reset_index(drop=True)
missing = sum(not (IMAGE_CACHE / Path(path).name).exists() for path in train_df.image_path)
assert len(train_df) == TRAINING_EXAMPLES
print(f'Строк для A1: {len(train_df)}; отсутствующих изображений в кэше: {missing}')
display(train_df[['type', 'question', 'answer', 'image_path']].head(3))

## 2. Датасет

Формат диалога и подготовку изображения задаёт `AutoProcessor` самой модели. Поэтому не нужно вручную нормализовать изображения или вставлять служебные токены: это снижает риск ошибок при смене архитектуры.

In [ ]:
def load_image(image_path):
    local_path = IMAGE_CACHE / Path(image_path).name
    if not local_path.exists():
        raise FileNotFoundError(f'Нет изображения в локальном кэше: {local_path}')
    return Image.open(local_path).convert('RGB')

class RussianVLMDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        return {'question': row.question, 'answer': row.answer, 'image_path': row.image_path}

dataset = RussianVLMDataset(train_df)
example = dataset[0]
print('Вопрос:', example['question'])
print('Ответ:', example['answer'][:180] + '...')
display(load_image(example['image_path']))

## 3. Загрузка InternVL3 и QLoRA

Модель загружается в 4-битном NF4-формате. LoRA обучает только небольшие добавочные матрицы в языковой части модели; визуальный энкодер остаётся замороженным. Важно: этот вариант использует `AutoProcessor`, а не старый `AutoTokenizer` с SentencePiece.

In [ ]:
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
# Один патч = 256 визуальных токенов. Без ограничения широкие изображения
# разбиваются на несколько патчей и не помещаются в MAX_LENGTH=1024.
processor.image_processor.max_patches = 1
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, low_cpu_mem_usage=True, quantization_config=quantization,
    device_map='auto', dtype=torch.float16,
).eval()
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Подготовка обучающих батчей

Collator формирует диалоги InternVL3, обрабатывает изображения и исключает текст вопроса из функции потерь.

In [ ]:
def user_message(item):
    return {'role': 'user', 'content': [
        {'type': 'image', 'image': load_image(item['image_path'])},
        {'type': 'text', 'text': item['question']},
    ]}

class InternVL3Collator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        conversations = []
        prompt_conversations = []
        images = []
        for item in features:
            user = user_message(item)
            conversations.append([user, {'role': 'assistant', 'content': [{'type': 'text', 'text': item['answer']}]}])
            prompt_conversations.append([user])
            images.append(load_image(item['image_path']))

        full_texts = [self.processor.apply_chat_template(c, tokenize=False) for c in conversations]
        prompt_texts = [self.processor.apply_chat_template(c, tokenize=False, add_generation_prompt=True)
                        for c in prompt_conversations]
        batch = self.processor(
            text=full_texts, images=images, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt',
            images_kwargs={'max_patches': 1},
        )
        prompt_batch = self.processor(
            text=prompt_texts, images=images, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt',
            images_kwargs={'max_patches': 1},
        )
        labels = batch.input_ids.clone()
        labels[batch.attention_mask == 0] = -100
        for i, prompt_length in enumerate(prompt_batch.attention_mask.sum(dim=1).tolist()):
            labels[i, :prompt_length] = -100
        batch['labels'] = labels
        return batch

collator = InternVL3Collator(processor)

## 5. Обучение A1

Ячейка запускает одну эпоху обучения. Чекпойнты сохраняются через каждые 50 шагов, а готовый LoRA-адаптер — в `models/a1_internvl3_2b_qlora_adapter`.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR), num_train_epochs=1,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_steps=8,
    logging_steps=5, save_strategy='steps', save_steps=50, save_total_limit=2,
    fp16=True, gradient_checkpointing=True, optim='paged_adamw_8bit',
    report_to='none', remove_unused_columns=False, seed=SEED,
)
trainer = Trainer(model=model, args=training_args, train_dataset=dataset, data_collator=collator)
print(f'Примерное число оптимизационных шагов: {len(dataset) // training_args.gradient_accumulation_steps}')
trainer.train()

model.save_pretrained(MODEL_DIR)
processor.save_pretrained(MODEL_DIR)
(RESULTS_DIR / 'a1_internvl3_training_config.json').write_text(json.dumps({
    'base_model': MODEL_ID, 'examples': len(dataset), 'lora_rank': 8,
    'epochs': 1, 'learning_rate': 2e-4, 'seed': SEED,
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Адаптер A1 сохранён в: {MODEL_DIR}')